# Speaker Identification — Embedding Enrollment & Matching

Smart Subtitle Glasses FYP — Sprint 1 speaker embedding experiment.

This notebook enrolls two speakers (Ayesha, Sabeeh) using `pyannote.audio` speaker
embeddings at four enrollment durations (5s/10s/15s/30s), tests identification
accuracy against held-out clips, calibrates the acceptance threshold, and matches
diarized speaker turns from a multi-speaker recording against the enrolled profiles.

**Note:** this is a cleaned reference version of the working pipeline. Re-run each
cell in order with your own audio files — Colab runtimes reset after ~90 minutes
idle, which wipes installed packages and in-memory variables (not files you've
separately downloaded).

## 1. Setup — install, authenticate, load embedding model

In [ ]:
!pip install -q "sympy==1.13.3"
!pip install -q pyannote.audio librosa

from getpass import getpass
from pyannote.audio import Model, Inference
from google.colab import files as colab_files
import re, json, numpy as np
from scipy.spatial.distance import cosine

HF_TOKEN = getpass("Paste your HuggingFace token: ")
embedding_model = Model.from_pretrained("pyannote/wespeaker-voxceleb-resnet34-LM", token=HF_TOKEN)
inference = Inference(embedding_model, window="whole")
print("✓ Embedding model ready")

## 2. Helper functions and experiment class

- `parse_filename` expects enrollment files named `Name_Ns.wav` (e.g. `Ayesha_5s.wav`)
- `ground_truth_from_filename` expects test files named `Name#.wav` (e.g. `Ayesha1.wav`),
  with unknown-speaker clips starting with `U` (e.g. `U1.wav`)

In [ ]:
def parse_filename(filename):
    match = re.match(r"([A-Za-z]+)_(\d+)s\.wav", filename, re.IGNORECASE)
    if not match:
        raise ValueError(f"'{filename}' doesn't match 'Name_Ns.wav'")
    name, dur = match.groups()
    return name, f"{dur}s"

def ground_truth_from_filename(fn):
    fn = re.sub(r"\s*\(\d+\)(?=\.\w+$)", "", fn)  # strips Colab's " (1)" duplicate suffix
    match = re.match(r"([A-Za-z]+)\d*\.wav", fn, re.IGNORECASE)
    name = match.group(1) if match else fn
    return "UNKNOWN" if name.upper().startswith("U") else name

class SpeakerIDExperiment:
    """Tracks per-duration speaker profiles for enrollment-length sensitivity testing."""
    def __init__(self, inference_engine):
        self.inference = inference_engine
        self.profiles = {}  # profiles[speaker][duration_label] = normalized embedding

    def enroll(self, speaker_name, duration_label, audio_path):
        emb = np.asarray(self.inference(audio_path)).flatten()
        emb = emb / np.linalg.norm(emb)
        self.profiles.setdefault(speaker_name, {})[duration_label] = emb
        print(f"✓ Enrolled {speaker_name} @ {duration_label}")

    def identify(self, test_audio_path, duration_label, threshold=0.40):
        test_emb = np.asarray(self.inference(test_audio_path)).flatten()
        test_emb = test_emb / np.linalg.norm(test_emb)
        scores = {s: round(1 - cosine(test_emb, d[duration_label]), 4)
                  for s, d in self.profiles.items() if duration_label in d}
        best = max(scores, key=scores.get)
        return {"test_audio": test_audio_path, "enrollment_duration": duration_label,
                "identified_speaker": "UNKNOWN" if scores[best] < threshold else best,
                "confidence": scores[best], "all_scores": scores,
                "is_unknown": scores[best] < threshold}

experiment = SpeakerIDExperiment(inference)
print("✓ Environment ready")

## 3. Enrollment

Upload 8 solo-speaker enrollment files: `Ayesha_5s.wav`, `Ayesha_10s.wav`,
`Ayesha_15s.wav`, `Ayesha_30s.wav`, `Sabeeh_5s.wav`, `Sabeeh_10s.wav`,
`Sabeeh_15s.wav`, `Sabeeh_30s.wav`.

In [ ]:
print("Upload your 8 enrollment files")
uploaded = colab_files.upload()

for filename in uploaded.keys():
    speaker_name, duration_label = parse_filename(filename)
    experiment.enroll(speaker_name, duration_label, filename)

print("\nEnrolled speakers:")
for speaker, durations in experiment.profiles.items():
    print(f"  {speaker}: {sorted(durations.keys())}")

## 4. Save enrolled embeddings permanently

Persists profiles to disk so a Colab disconnect never costs a re-enrollment again —
download the JSON and keep it; re-upload and reload it in any future session
instead of re-running enrollment from raw audio.

In [ ]:
def save_profiles(experiment, path="enrolled_speakers.json"):
    serializable = {
        speaker: {dur: emb.tolist() for dur, emb in durations.items()}
        for speaker, durations in experiment.profiles.items()
    }
    with open(path, "w") as f:
        json.dump(serializable, f)
    print(f"✓ Saved {path}")

def load_profiles(path="enrolled_speakers.json"):
    with open(path) as f:
        raw = json.load(f)
    return {speaker: {dur: np.array(vec) for dur, vec in durations.items()}
            for speaker, durations in raw.items()}

save_profiles(experiment)
colab_files.download("enrolled_speakers.json")

# In a future session, restore instead of re-enrolling:
# uploaded = colab_files.upload()  # pick enrolled_speakers.json
# experiment.profiles = load_profiles("enrolled_speakers.json")

## 5. Identification testing

Upload held-out test clips: 5 per enrolled speaker (e.g. `Ayesha1.wav`–`Ayesha5.wav`,
`Sabeeh1.wav`–`Sabeeh5.wav`) plus several unknown-speaker clips (`U1.wav`, `U2.wav`, ...).

In [ ]:
import pandas as pd

print("Upload your test clips")
test_uploaded = colab_files.upload()
test_files = list(test_uploaded.keys())

all_results = []
for duration_label in ["5s", "10s", "15s", "30s"]:
    for test_file in test_files:
        result = experiment.identify(test_file, duration_label, threshold=0.40)
        result["ground_truth"] = ground_truth_from_filename(test_file)
        result["is_correct"] = result["identified_speaker"] == result["ground_truth"]
        all_results.append(result)

results_df = pd.DataFrame(all_results)

print("\nOverall accuracy by enrollment duration:")
print(results_df.groupby("enrollment_duration")["is_correct"].mean())
print("\nUnknown-speaker rejection rate:")
print(results_df[results_df["ground_truth"]=="UNKNOWN"].groupby("enrollment_duration")["is_correct"].mean())
print("\nKnown-speaker accuracy only:")
print(results_df[results_df["ground_truth"]!="UNKNOWN"].groupby("enrollment_duration")["is_correct"].mean())

results_df.to_csv("identification_results.csv", index=False)
colab_files.download("identification_results.csv")

## 6. Threshold calibration note

Initial default threshold (0.70) was too conservative. Calibrating against this
held-out test set showed **0.40** gives 100% known-speaker accuracy and 100%
unknown-speaker rejection on this pilot's data (see `reports/` for the full
before/after analysis). Re-validate this threshold as more speakers and stress
conditions (noise, distance, Urdu) are added.

## 7. Diarization + embedding matching (integration step)

Takes a multi-speaker recording, runs it through the diarization pipeline, embeds
each diarized turn, and matches it against the enrolled profiles — the step that
combines diarization output with speaker identification.

Requires `pipeline` (the diarization pipeline from your diarization notebook) to
already be loaded in this session.

In [ ]:
from pyannote.core import Segment

def match_diarized_speakers(audio_path, duration_label="30s", threshold=0.40):
    diarization_result = pipeline(audio_path)
    results = []
    for turn, _, dia_label in diarization_result.speaker_diarization.itertracks(yield_label=True):
        if turn.duration < 1.0:
            continue  # too short for a reliable embedding
        seg_emb = np.asarray(inference.crop(audio_path, turn)).flatten()
        seg_emb = seg_emb / np.linalg.norm(seg_emb)
        scores = {name: round(1 - cosine(seg_emb, d[duration_label]), 4)
                  for name, d in experiment.profiles.items() if duration_label in d}
        best = max(scores, key=scores.get)
        identified = best if scores[best] >= threshold else "UNKNOWN"
        results.append({"start": round(turn.start, 2), "end": round(turn.end, 2),
                         "diarization_label": dia_label, "identified_speaker": identified,
                         "confidence": scores[best]})
        print(f"{turn.start:.1f}s\u2013{turn.end:.1f}s [{dia_label}] \u2192 {identified} (conf={scores[best]:.3f})")
    return results

# match_diarized_speakers("your_conversation.wav", duration_label="30s")